In [ ]:
# -*- coding: utf-8 -*-
"""
Compensação térmica com Autoencoder Residual Condicionado por Temperatura
+ Aplicação global ponto a ponto
+ Classificação multiclasse com split térmico sem overlap
+ Métricas completas
+ Gráfico principal
Autor: Luiz Eduardo Abdala José
"""

import re, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    r2_score, mean_squared_error, mean_absolute_error,
    accuracy_score, f1_score
)
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore", category=UserWarning)


# ========= PARÂMETROS =========
ARQ_BASE = "base-completo--.pkl"

REF_TEMP = 30
FREQ_MIN_KHZ = 40
FREQ_MAX_KHZ = 50
SMOOTH_WIN = 5

ALPHA_COMP = 0.85

AE_EPOCHS = 800
AE_BATCH_SIZE = 16
AE_LR = 1e-3
AE_LATENT_DIM = 64

RF_CLASSIF_PARAMS = dict(
    n_estimators=400,
    max_depth=10,
    min_samples_split=4,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)


# ========= FUNÇÕES =========
def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None


def get_freq_columns(df, fmin_khz, fmax_khz):
    cols, freqs = [], []

    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None and fmin_khz <= f / 1e3 <= fmax_khz:
            cols.append(c)
            freqs.append(f)

    order = np.argsort(freqs)
    return [cols[i] for i in order], np.array(freqs, float)[order]


def add_extra_features(X):
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True)
    amp = (X.max(axis=1) - X.min(axis=1)).reshape(-1, 1)
    return np.hstack([X, mu, sd, amp])


def add_temp_feature(X_aug, temp_vec):
    return np.hstack([X_aug, np.asarray(temp_vec).reshape(-1, 1)])


def moving_average(arr, win):
    if win <= 1 or win % 2 == 0:
        return arr.copy()

    pad = win // 2
    arr_pad = np.pad(arr, (pad, pad), mode="edge")
    kernel = np.ones(win) / win
    smooth = np.convolve(arr_pad, kernel, mode="valid")

    return smooth[:len(arr)]


def spectral_entropy(x):
    x = np.asarray(x, float)
    p = np.abs(x) ** 2
    s = p.sum()

    if s <= 0:
        return 0.0

    p /= s
    return float(-np.sum(p * np.log2(p + 1e-12)))


def calc_curve_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)

    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    corr = float(np.corrcoef(y_true, y_pred)[0, 1])

    num = float(np.dot(y_true, y_pred))
    den = float(np.linalg.norm(y_true) * np.linalg.norm(y_pred) + 1e-12)
    sam_deg = float(np.degrees(np.arccos(np.clip(num / den, -1, 1))))

    nrmse = rmse / (y_true.max() - y_true.min() + 1e-12)

    diff = y_true - y_pred
    rmsd = float(np.sqrt(np.mean((diff - diff.mean()) ** 2)))
    ccdm = float(1 - corr)

    e_true = np.sum(y_true ** 2)
    e_pred = np.sum(y_pred ** 2)
    eo = float(
        2 * np.sum(np.minimum(y_true ** 2, y_pred ** 2))
        / (e_true + e_pred + 1e-12)
    )

    ent_true = spectral_entropy(y_true)
    ent_pred = spectral_entropy(y_pred)

    return dict(
        R2=r2,
        RMSE=rmse,
        MAE=mae,
        Corr=corr,
        SAM_deg=sam_deg,
        NRMSE=nrmse,
        RMSD=rmsd,
        CCDM=ccdm,
        EnergyOverlap=eo,
        EntropyTrue=ent_true,
        EntropyPred=ent_pred,
        EntropyDiff=abs(ent_true - ent_pred)
    )


# ========= AUTOENCODER RESIDUAL =========
class ResidualAutoencoder(nn.Module):
    def __init__(self, n_in, n_out, latent_dim=64):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(n_in, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, latent_dim),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, n_out)
        )

    def forward(self, x):
        z = self.encoder(x)
        delta = self.decoder(z)
        return delta


def train_autoencoder_residual(
    X_input,
    Y_target,
    epochs=800,
    batch_size=16,
    lr=1e-3,
    latent_dim=64
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\nDispositivo usado no Autoencoder: {device}")

    sx = StandardScaler()
    sy = StandardScaler()

    Xs = sx.fit_transform(X_input)
    Ys = sy.fit_transform(Y_target)

    X_tensor = torch.tensor(Xs, dtype=torch.float32)
    Y_tensor = torch.tensor(Ys, dtype=torch.float32)

    loader = DataLoader(
        TensorDataset(X_tensor, Y_tensor),
        batch_size=batch_size,
        shuffle=True
    )

    model = ResidualAutoencoder(
        n_in=X_input.shape[1],
        n_out=Y_target.shape[1],
        latent_dim=latent_dim
    ).to(device)

    opt = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=1e-5
    )

    loss_fn = nn.MSELoss()

    model.train()

    for ep in range(epochs):
        total_loss = 0.0

        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            opt.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()

            total_loss += loss.item()

        if (ep + 1) % 100 == 0:
            print(
                f"Epoch {ep+1:4d}/{epochs} | "
                f"loss = {total_loss / len(loader):.6f}"
            )

    return model, sx, sy, device


def predict_autoencoder_residual(model, sx, sy, device, X_input):
    model.eval()

    Xs = sx.transform(X_input)
    X_tensor = torch.tensor(Xs, dtype=torch.float32).to(device)

    with torch.no_grad():
        pred_scaled = model(X_tensor).cpu().numpy()

    return sy.inverse_transform(pred_scaled)


# ========= SCRIPT =========
if __name__ == "__main__":

    timings = {}

    # 1) CARREGAMENTO
    t0 = time.time()

    df = pd.read_pickle(ARQ_BASE)

    fcols, fhz = get_freq_columns(
        df,
        FREQ_MIN_KHZ,
        FREQ_MAX_KHZ
    )

    fhz_khz = fhz / 1e3

    df_sem = df[df["falha"] == 0].copy()

    timings["load"] = time.time() - t0

    print(f"Amostras sem falha: {len(df_sem)} | total: {len(df)}")
    print(f"Faixa usada: {FREQ_MIN_KHZ}-{FREQ_MAX_KHZ} kHz")
    print(f"Número de pontos de frequência: {len(fcols)}")

    # 2) REFERÊNCIA
    t0 = time.time()

    pool_ref = df_sem.loc[
        np.isclose(df_sem["temperatura_c"], REF_TEMP),
        fcols
    ].to_numpy(float)

    if len(pool_ref) == 0:
        raise ValueError(
            f"Nenhuma amostra sem falha encontrada em REF_TEMP = {REF_TEMP}°C"
        )

    y_ref = np.median(pool_ref, axis=0)

    timings["reference"] = time.time() - t0

    # 3) TREINO AUTOENCODER RESIDUAL
    t0 = time.time()

    X_sem = df_sem[fcols].to_numpy(float)
    T_sem = df_sem["temperatura_c"].to_numpy(float)

    # O alvo é a correção necessária para levar o sinal saudável até a referência.
    Y_target = y_ref[None, :] - X_sem

    X_aug_sem = add_extra_features(X_sem)
    X_comp_sem = add_temp_feature(X_aug_sem, T_sem)

    ae_comp, sx_ae, sy_ae, device = train_autoencoder_residual(
        X_comp_sem,
        Y_target,
        epochs=AE_EPOCHS,
        batch_size=AE_BATCH_SIZE,
        lr=AE_LR,
        latent_dim=AE_LATENT_DIM
    )

    timings["train_comp_AE"] = time.time() - t0

    # 4) APLICA COMPENSAÇÃO COM AUTOENCODER
    t0 = time.time()

    X_all = df[fcols].to_numpy(float)
    T_all = df["temperatura_c"].to_numpy(float)

    X_aug_all = add_extra_features(X_all)
    X_comp_all = add_temp_feature(X_aug_all, T_all)

    Delta_hat = predict_autoencoder_residual(
        ae_comp,
        sx_ae,
        sy_ae,
        device,
        X_comp_all
    )

    # Compensação residual com fator de segurança.
    Y_hat = X_all + ALPHA_COMP * Delta_hat

    for i in range(len(Y_hat)):
        Y_hat[i] = moving_average(Y_hat[i], SMOOTH_WIN)

    df_comp = df.copy()
    df_comp[fcols] = Y_hat

    timings["apply_comp_AE"] = time.time() - t0

    # 5) SPLIT SEM OVERLAP TÉRMICO
    t0 = time.time()

    temps = sorted(df["temperatura_c"].unique())

    temps_train = temps[::2]
    temps_test = temps[1::2]

    df_train = df_comp[df_comp["temperatura_c"].isin(temps_train)]
    df_test = df_comp[df_comp["temperatura_c"].isin(temps_test)]

    X_train = df_train[fcols].to_numpy(float)
    y_train = df_train["falha"].to_numpy(int)

    X_test = df_test[fcols].to_numpy(float)
    y_test = df_test["falha"].to_numpy(int)

    timings["split"] = time.time() - t0

    print(f"\nTemperaturas de treino: {temps_train}")
    print(f"Temperaturas de teste : {temps_test}")

    # 6) CLASSIFICAÇÃO
    t0 = time.time()

    clf = RandomForestClassifier(**RF_CLASSIF_PARAMS)
    clf.fit(X_train, y_train)

    timings["train_clf"] = time.time() - t0

    t0 = time.time()

    y_pred = clf.predict(X_test)

    timings["predict_clf"] = time.time() - t0

    cm = confusion_matrix(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro")

    # 7) MÉTRICAS DA CURVA EXEMPLO
    print("\n===== ÍNDICES DISPONÍVEIS EM df_test =====")

    for i, idx in enumerate(df_test.index):
        temp = df_test.loc[idx, "temperatura_c"]
        falha = df_test.loc[idx, "falha"]
        print(
            f"Posição {i:3d} | "
            f"Índice real: {idx} | "
            f"Temperatura: {temp}°C | "
            f"Falha: {falha}"
        )

    pos_show = 17 if len(df_test) > 17 else 0
    idx_show = df_test.index[pos_show]

    curve_metrics = calc_curve_metrics(
        y_ref,
        df_comp.loc[idx_show, fcols].to_numpy(float)
    )

    print("\n============== MÉTRICAS DA COMPENSAÇÃO AE ==============")
    print(f"Curva exemplo: posição {pos_show} | índice {idx_show}")
    print(f"Temperatura original: {df.loc[idx_show, 'temperatura_c']}°C")
    print(f"Falha: {df.loc[idx_show, 'falha']}")

    for k, v in curve_metrics.items():
        print(f"{k:20s}: {v:.6f}")

    print("\n==================== CLASSIFICAÇÃO ====================")
    print("Matriz de confusão:")
    print(cm)

    print(f"\nACC   = {acc:.4f}")
    print(f"F1    = {macro_f1:.4f}")

    print("\nRelatório completo:")
    print(classification_report(y_test, y_pred, digits=4))

    print("\n==================== TEMPOS (s) ====================")
    for k, v in timings.items():
        print(f"{k:20s}: {v:.4f}")

    # 8) GRÁFICO FINAL
    print("\n🔹 Gerando gráfico de exemplo...")

    plt.rcParams.update({
        "font.size": 10,
        "text.usetex": False,
        "font.family": "Times New Roman"
    })

    plt.figure(figsize=(8, 4))

    plt.plot(
        fhz_khz,
        y_ref,
        "--",
        c="black",
        lw=0.5,
        label=f"Referência {REF_TEMP}°C"
    )

    plt.plot(
        fhz_khz,
        df.loc[idx_show, fcols],
        c="tab:red",
        alpha=0.6,
        label=f"Original {df.loc[idx_show, 'temperatura_c']}°C"
    )

    plt.plot(
        fhz_khz,
        df_comp.loc[idx_show, fcols],
        c="tab:blue",
        lw=2,
        label=f"Compensado AE {df.loc[idx_show, 'temperatura_c']}°C"
    )

    plt.title(
        f"Autoencoder Residual — {FREQ_MIN_KHZ}-{FREQ_MAX_KHZ} kHz"
    )
    plt.xlabel("Frequência (kHz)")
    plt.ylabel("Parte real da impedância")

    plt.legend(
        frameon=True,
        facecolor="white",
        edgecolor="none"
    )

    plt.grid(alpha=0.0)
    plt.tight_layout()
    plt.show()

In [ ]:
# 8) GRÁFICO FINAL — FALHA 0, 1 E 2
print("\n🔹 Gerando gráfico com falha 0, 1 e 2...")

plt.rcParams.update({
    "font.size": 10,
    "text.usetex": False,
    "font.family": "Times New Roman"
})

# Temperatura que você quer comparar
# Aqui usa a mesma temperatura do idx_show escolhido antes
TEMP_SHOW = df.loc[idx_show, "temperatura_c"]

# Classes de dano/falha que serão plotadas
FALHAS_SHOW = [0, 1, 2]

# Cores para cada falha
cores = {
    0: "tab:green",
    1: "tab:orange",
    2: "tab:red"
}

plt.figure(figsize=(9, 4.5))

# Referência saudável em REF_TEMP
plt.plot(
    fhz_khz,
    y_ref,
    "--",
    c="black",
    lw=1.0,
    label=f"Referência saudável {REF_TEMP}°C"
)

for falha in FALHAS_SHOW:

    # Pega somente as amostras daquela falha
    sub = df[df["falha"] == falha]

    if len(sub) == 0:
        print(f"Nenhuma amostra encontrada para falha {falha}")
        continue

    # Escolhe a curva daquela falha com temperatura mais próxima de TEMP_SHOW
    idx_falha = (sub["temperatura_c"] - TEMP_SHOW).abs().idxmin()

    temp_real = df.loc[idx_falha, "temperatura_c"]

    # Curva original
    plt.plot(
        fhz_khz,
        df.loc[idx_falha, fcols].to_numpy(float),
        color=cores[falha],
        alpha=0.35,
        lw=1.0,
        linestyle="-",
        label=f"Original Falha {falha} — {temp_real}°C"
    )

    # Curva compensada
    plt.plot(
        fhz_khz,
        df_comp.loc[idx_falha, fcols].to_numpy(float),
        color=cores[falha],
        alpha=1.0,
        lw=2.0,
        linestyle="-",
        label=f"Compensado AE Falha {falha} — {temp_real}°C"
    )

plt.title(
    f"Autoencoder Residual — Falhas 0, 1 e 2 — {FREQ_MIN_KHZ}-{FREQ_MAX_KHZ} kHz"
)

plt.xlabel("Frequência (kHz)")
plt.ylabel("Parte real da impedância")

plt.legend(
    frameon=True,
    facecolor="white",
    edgecolor="none",
    fontsize=8
)

plt.grid(alpha=0.0)
plt.tight_layout()
plt.show()

In [ ]:
# 8) GRÁFICOS FINAIS — UM GRÁFICO PARA CADA FALHA
print("\n🔹 Gerando gráficos separados para falha 0, 1 e 2...")

plt.rcParams.update({
    "font.size": 10,
    "text.usetex": False,
    "font.family": "Times New Roman"
})

# Temperatura que você quer comparar
# Aqui usa a mesma temperatura da curva exemplo idx_show
TEMP_SHOW = df.loc[idx_show, "temperatura_c"]

# Se quiser escolher manualmente, descomente:
# TEMP_SHOW = 70

FALHAS_SHOW = [0, 1, 2]

cores = {
    0: "tab:green",
    1: "tab:orange",
    2: "tab:red"
}

nomes_falha = {
    0: "Sem falha",
    1: "Dano 1",
    2: "Dano 2"
}

for falha in FALHAS_SHOW:

    # Seleciona somente as amostras da falha atual
    sub = df[df["falha"] == falha]

    if len(sub) == 0:
        print(f"Nenhuma amostra encontrada para falha {falha}")
        continue

    # Escolhe a curva com temperatura mais próxima de TEMP_SHOW
    idx_falha = (sub["temperatura_c"] - TEMP_SHOW).abs().idxmin()

    temp_real = df.loc[idx_falha, "temperatura_c"]

    print(
        f"Falha {falha} | índice {idx_falha} | "
        f"temperatura escolhida: {temp_real}°C"
    )

    plt.figure(figsize=(8, 4))

    # Referência saudável
    plt.plot(
        fhz_khz,
        y_ref,
        "--",
        c="black",
        lw=1.0,
        label=f"Referência saudável {REF_TEMP}°C"
    )

    # Curva original da falha atual
    plt.plot(
        fhz_khz,
        df.loc[idx_falha, fcols].to_numpy(float),
        color=cores[falha],
        alpha=0.55,
        lw=1.2,
        label=f"Original — {nomes_falha[falha]} — {temp_real}°C"
    )

    # Curva compensada da falha atual
    plt.plot(
        fhz_khz,
        df_comp.loc[idx_falha, fcols].to_numpy(float),
        color="tab:blue",
        alpha=1.0,
        lw=2.0,
        label=f"Compensado AE — {nomes_falha[falha]} — {temp_real}°C"
    )

    plt.title(
        f"Autoencoder Residual — {nomes_falha[falha]} — "
        f"{FREQ_MIN_KHZ}-{FREQ_MAX_KHZ} kHz"
    )

    plt.xlabel("Frequência (kHz)")
    plt.ylabel("Parte real da impedância")

    plt.legend(
        frameon=True,
        facecolor="white",
        edgecolor="none",
        fontsize=8
    )

    plt.grid(alpha=0.0)
    plt.tight_layout()
    plt.show()